# C6-pytorch — Practice p09 — Solution


Each requested affine expression becomes one row of `W_a`, with its
constant in the matching bias entry.  For the detector, the single
score is exactly $3x_1-x_2-2$; thresholding that score implements the
inclusive inequality.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


P = torch.tensor([[0.0, 0.0], [1.0, 2.0], [-1.0, 1.0], [2.0, -0.5]])

def target_a(x):
    return torch.stack([x[:, 0] - x[:, 1],
                        2 * x[:, 1] + 1,
                        x[:, 0] + x[:, 1] - 3], dim=1)

def target_b(x):
    return (3 * x[:, 0] - x[:, 1] - 2 >= 0).to(x.dtype)


W_a = torch.tensor([[1.0, -1.0], [0.0, 2.0], [1.0, 1.0]])
b_a = torch.tensor([0.0, 1.0, -3.0])
layer_a = DenseLayer(W_a, b_a)
y_a = layer_a(P)
gap_a = (y_a - target_a(P)).abs().max().item()

detector = nn.Sequential(
    DenseLayer(torch.tensor([[3.0, -1.0]]), torch.tensor([-2.0])),
    ThresholdGate(),
)
y_b = detector(P)
gap_b = (y_b.ravel() - target_b(P)).abs().max().item()

y_a, gap_a, y_b, gap_b


### Answer check


In [ ]:
assert torch.equal(y_a, torch.tensor([[0.0, 1.0, -3.0],
                                      [-1.0, 5.0, 0.0],
                                      [-2.0, 3.0, -3.0],
                                      [2.5, 0.0, -1.5]]))
assert torch.equal(y_b.ravel(), torch.tensor([0.0, 0.0, 0.0, 1.0]))
assert gap_a == 0.0 and gap_b == 0.0
